# Generation of samples

## Experiment output folder

In [ ]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Model loading

In [ ]:
import torch
from torch.utils.data import DataLoader

from modules.model import VariationalAutoencoder
from modules.dataset import LogMinMaxScale, EnsembleDataset

checkpoint = torch.load(output_dir / 'model.pt', map_location='cpu')

## Normalization
min_value = checkpoint['MINVAL']
max_value = checkpoint['MAXVAL']
scale     = checkpoint['SCALE']
transform = LogMinMaxScale(min_value, max_value, scale)

# Model
model = VariationalAutoencoder(checkpoint['LATENT_DIM'], in_shape=checkpoint['IN_SHAPE'])
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
device = torch.device('cpu')
latent_dim = checkpoint['LATENT_DIM']

## Load train dataset

In [ ]:
import xarray as xr

fname = checkpoint['FNAME_TRAIN']
varkey = checkpoint['VARKEY']

## Loading raw data and normaliation
ds = xr.open_dataset(fname)
da = ds[varkey]

dataset = EnsembleDataset(da, transform)
loader  = DataLoader(dataset, batch_size=12,shuffle=False)

## Sampling strategies: choose one

In [ ]:
#
# Number of sampling to be generated
#
N = 4096

Three sampling strategies are provided below. Select just one (recommened: _3. Full Gaussian approximation_)

### 1. Standard VAE prior

In [ ]:
# Standard normal samples
z = torch.randn(N, latent_dim)

### 2. Diagonal approximation

In [ ]:
# ------------------------------------------------------------
# 1. Encode the dataset and collect posterior samples
# ------------------------------------------------------------

model.eval()

mus = []
logvars = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)

        mu, logvar = model.encode(batch)

        mus.append(mu.cpu())
        logvars.append(logvar.cpu())

mu = torch.cat(mus, dim=0)          # [N_samples, latent_dim]
logvar = torch.cat(logvars, dim=0)  # [N_samples, latent_dim]

sigma = torch.exp(0.5 * logvar)

# One sample from q(z|x) for each input sample
z_train = mu + sigma * torch.randn_like(mu)

# ------------------------------------------------------------
# 2. Estimate the diagonal Gaussian parameters
# ------------------------------------------------------------

z_mean = z_train.mean(dim=0)
z_std = z_train.std(dim=0)

# ------------------------------------------------------------
# 3. Generate new latent samples
# ------------------------------------------------------------

# Standard normal samples
z = torch.randn(N, latent_dim)

# Transform N(0,I) -> N(z_mean, diag(z_std^2))
z = z * z_std + z_mean

### 3. Full Gaussian approximation

In [ ]:
# ------------------------------------------------------------
# 1. Encode the dataset and collect posterior samples
# ------------------------------------------------------------

model.eval()

mus = []
logvars = []

with torch.no_grad():
    for batch in loader:
        batch = batch.to(device)

        mu, logvar = model.encode(batch)
        #z = model.reparameterize(mu,logvar)

        mus.append(mu.cpu())
        logvars.append(logvar.cpu())

mu = torch.cat(mus, dim=0)          # [N_samples, latent_dim]
logvar = torch.cat(logvars, dim=0)  # [N_samples, latent_dim]

sigma = torch.exp(0.5 * logvar)

# One sample from q(z|x) for each input sample
z_train = mu + sigma * torch.randn_like(mu)

# ------------------------------------------------------------
# 2. Estimate the Gaussian parameters
# ------------------------------------------------------------

z_mean = z_train.mean(dim=0)
cov_z = torch.cov(z_train.T)

# Small jitter for numerical stability
eps = 1e-6
cov_z = cov_z + eps * torch.eye(cov_z.shape[0])

L = torch.linalg.cholesky(cov_z)

# ------------------------------------------------------------
# 3. Generate new latent samples
# ------------------------------------------------------------

# Standard normal samples
eps = torch.randn(N, latent_dim)

# Transform N(0,I) -> N(z_mean, Σ)
z = z_mean + eps @ L.T

## Decode

In [ ]:
model.eval()

with torch.no_grad():
    z = z.to(device)

    x_generated = model.decode(z)

    # Back to original concentration units
    x_generated_raw = transform.invert(x_generated).squeeze()

    # Detach and move to CPU once right after creation
    x_generated_raw = x_generated_raw.detach().cpu()
    z = z.detach().cpu()

## Save samples

In [ ]:
data_vars = {
    "samples": (("ens", "lat", "lon"), x_generated_raw.numpy()),
    "mean": (("lat","lon"), x_generated_raw.mean(dim=0).numpy()),
    "z": (("ens", "latent_dim"), z.numpy()),
}
ds = xr.Dataset(
    data_vars,
    coords={"lat": ds.lat,"lon": ds.lon},
)
# Optional: Add metadata attributes to latitude and longitude
ds["lat"].attrs = {"units": "degrees_north", "standard_name": "latitude"}
ds["lon"].attrs = {"units": "degrees_east", "standard_name": "longitude"}
ds.to_netcdf(output_dir / "prior.nc")